# Bank Term Deposit Prediction

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"

# Data Understanding

In [ ]:
df = pd.read_csv("dataset/bank.csv")

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df['deposit'].value_counts()

# Data Cleaning

In [ ]:
df.isnull().sum().sort_values(ascending=False)

In [ ]:
df.duplicated().sum()

In [ ]:
df.drop_duplicates(inplace=True)
df.reset_index(drop=True,inplace=True)

In [ ]:
df = df.dropna(subset=['deposit'])

In [ ]:
df['deposit'] = df['deposit'].map({'yes':1,'no':0})

In [ ]:
df.drop('duration',axis=1,inplace=True)

In [ ]:
df.shape

In [ ]:
df.isnull().sum().sort_values(ascending=False).head(10)

# Train Test Split

In [ ]:
X = df.drop('deposit',axis = 1)
y = df['deposit']

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.2,shuffle=True,random_state=42,stratify=y)

In [ ]:
X_train.reset_index(drop=True,inplace=True)
X_test.reset_index(drop=True,inplace=True)

y_train.reset_index(drop=True,inplace=True)
y_test.reset_index(drop=True,inplace=True)

In [ ]:
y_train.value_counts(normalize=True)

# Missing Values

In [ ]:
X_train.isnull().sum().sort_values(ascending=False).head(10)

In [ ]:
from sklearn.impute import SimpleImputer

In [ ]:
X_train[['balance', 'campaign', 'pdays', 'previous', 'day']].skew()

In [ ]:
median = ['balance', 'campaign', 'pdays', 'previous']
impute = SimpleImputer(strategy='median')
for col in median:
    X_train[col] = impute.fit_transform(X_train[[col]])
    X_test[col] = impute.transform(X_test[[col]])

In [ ]:
mean = ['day']
impute = SimpleImputer(strategy='mean')
for col in mean:
    X_train[col] = impute.fit_transform(X_train[[col]])
    X_test[col] = impute.transform(X_test[[col]])

In [ ]:
mode = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month']
impute = SimpleImputer(strategy='most_frequent')
for col in mode:
    X_train[[col]] = impute.fit_transform(X_train[[col]])
    X_test[[col]] = impute.transform(X_test[[col]])

In [ ]:
X_train.isnull().sum().sum() , X_test.isnull().sum().sum()

In [ ]:
X_train.duplicated().sum() , X_test.duplicated().sum()

# Data Visualization (EDA)

In [ ]:
df_train = pd.concat([X_train,y_train],axis=1)

In [ ]:
sns.countplot(data=df_train,x='deposit')
plt.show()

In [ ]:
top_cols = df_train.corr(numeric_only=True)['deposit'].abs().sort_values(ascending=False).head(12).index
plt.figure(figsize=(12,8))
sns.heatmap(df_train[top_cols].corr(),annot=True,cmap='Blues',fmt='0.2f')
plt.show()

In [ ]:
num_cols = ['pdays', 'previous', 'campaign', 'balance', 'day', 'age']
plt.figure(figsize=(18,8))
for n , col in enumerate(num_cols):
    plt.subplot(2,3,n+1)
    plt.hist(df_train[col],bins=30)
    plt.title(col)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(18,8))
for n , col in enumerate(num_cols):
    plt.subplot(2,3,n+1)
    sns.boxplot(data=df_train,x=col)
plt.tight_layout()
plt.show()

In [ ]:
sns.scatterplot(data=df_train,x='pdays',y='previous',hue='deposit')
plt.show()

In [ ]:
plt.figure(figsize=(20,5))
for n , c in enumerate(df_train['education'].unique()):
    plt.subplot(1,4,n+1)
    df_train[df_train['education']==c]['deposit'].value_counts().reindex([0,1],fill_value=0).plot(kind = "bar")
    plt.title(str(c))
    plt.xticks([0,1],["0","1"])
    plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(20,5))
for n , c in enumerate(df_train['poutcome'].unique()):
    plt.subplot(1,4,n+1)
    df_train[df_train['poutcome']==c]['deposit'].value_counts().reindex([0,1],fill_value=0).plot(kind = "bar")
    plt.title(str(c))
    plt.xticks([0,1],["0","1"])
    plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(20,5))
for n , c in enumerate(df_train['marital'].unique()):
    plt.subplot(1,3,n+1)
    df_train[df_train['marital']==c]['deposit'].value_counts().reindex([0,1],fill_value=0).plot(kind = "bar")
    plt.title(str(c))
    plt.xticks([0,1],["0","1"])
    plt.tight_layout()
plt.show()

# Data Encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder , OrdinalEncoder , OneHotEncoder , StandardScaler

In [ ]:
X_train.select_dtypes(exclude='number').nunique()

In [ ]:
binary_cols = ['default', 'housing', 'loan']
lb = LabelEncoder()
for col in binary_cols:
    X_train[col] = lb.fit_transform(X_train[col])
    X_test[col] = lb.transform(X_test[col])

In [ ]:
cat_cols = ['job', 'marital', 'education', 'contact', 'month', 'poutcome']

In [ ]:
onehot = OneHotEncoder(handle_unknown='ignore')
onehot.fit(X_train[cat_cols])

In [ ]:
temp = onehot.transform(X_train[cat_cols]).toarray()
temp = pd.DataFrame(temp,columns=onehot.get_feature_names_out(cat_cols))
X_train = pd.concat([X_train,temp],axis=1)
X_train.drop(cat_cols,axis=1,inplace=True)

In [ ]:
temp = onehot.transform(X_test[cat_cols]).toarray()
temp = pd.DataFrame(temp,columns=onehot.get_feature_names_out(cat_cols))
X_test = pd.concat([X_test,temp],axis=1)
X_test.drop(cat_cols,axis=1,inplace=True)

In [ ]:
X_train.shape , X_test.shape

# Feature Scaling

In [ ]:
X_train[['campaign', 'previous']].skew()

In [ ]:
log_cols = ['campaign', 'previous']
for col in log_cols:
    X_train[col] = np.log1p(X_train[col])
    X_test[col] = np.log1p(X_test[col])

In [ ]:
scale_cols = ['age', 'balance', 'day', 'campaign', 'pdays', 'previous']
scaler = StandardScaler()
X_train[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_test[scale_cols] = scaler.transform(X_test[scale_cols])

In [ ]:
X_train.head()

In [ ]:
X_train.isnull().sum().sum() , X_test.isnull().sum().sum()

In [ ]:
list(X_train.columns) == list(X_test.columns)

# Build Machine Learning Models

In [ ]:
from tqdm import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier, AdaBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score
from sklearn.metrics import confusion_matrix, classification_report, RocCurveDisplay, PrecisionRecallDisplay

In [ ]:
models = {
    "Logistic Regression":LogisticRegression(max_iter=1000),
    "KNN":KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes":GaussianNB(),
    "SVM":SVC(kernel='rbf',C=1),
    "Decision Tree":DecisionTreeClassifier(max_depth=10,random_state=42),
    "Random Forest":RandomForestClassifier(n_estimators=100,max_depth=10,random_state=42),
    "ExtraTrees":ExtraTreesClassifier(n_estimators=100,random_state=42),
    "Gradient Boosting":GradientBoostingClassifier(n_estimators=100,learning_rate=0.1,random_state=42),
    "AdaBoost":AdaBoostClassifier(n_estimators=50,random_state=42),
    "XGBoost":XGBClassifier(n_estimators=100,learning_rate=0.1,random_state=42,verbosity=0),
    "LightGBM":LGBMClassifier(n_estimators=100,learning_rate=0.1,random_state=42,verbose=-1)
}

In [ ]:
def get_fit_data(model_name):
    if model_name in ['SVM','KNN'] and len(X_train) > 20000:
        return X_train[:20000] , y_train[:20000]
    return X_train , y_train

In [ ]:
def classification_metrics(model, X, y_actual):
    y_pred = model.predict(X)
    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X)[:,1]
    else:
        y_score = model.decision_function(X)
    Accuracy = accuracy_score(y_actual, y_pred)
    Precision = precision_score(y_actual, y_pred, zero_division=0)
    Recall = recall_score(y_actual, y_pred)
    F1 = f1_score(y_actual, y_pred)
    AUC = roc_auc_score(y_actual, y_score)
    return Accuracy, Precision, Recall, F1, AUC

In [ ]:
errors={}
for model_name , model in tqdm(models.items()):
    X_fit , y_fit = get_fit_data(model_name)
    model.fit(X_fit,y_fit)
    Acc_Train, Pre_Train, Rec_Train, F1_Train, AUC_Train = classification_metrics(model, X_fit, y_fit)
    Acc_Test, Pre_Test, Rec_Test, F1_Test, AUC_Test = classification_metrics(model, X_test, y_test)

    errors[model_name] = {
        "Accuracy Train":Acc_Train,
        "Accuracy Test":Acc_Test,
        "Precision Train":Pre_Train,
        "Precision Test":Pre_Test,
        "Recall Train":Rec_Train,
        "Recall Test":Rec_Test,
        "F1 Train":F1_Train,
        "F1 Test":F1_Test,
        "ROC AUC Train":AUC_Train,
        "ROC AUC Test":AUC_Test
    }

# Models Comparison (Before Tuning)

In [ ]:
errors_df = pd.DataFrame(errors).T
errors_df

In [ ]:
cols = [
    ['Accuracy Train', 'Accuracy Test'],
    ['Precision Train', 'Precision Test'],
    ['Recall Train', 'Recall Test'],
    ['F1 Train', 'F1 Test'],
    ['ROC AUC Train', 'ROC AUC Test']
]

plt.figure(figsize=(20,22))
for i, col in enumerate(cols):
    ax = plt.subplot(3, 2, i+1)
    errors_df[col].plot.bar(ax=ax)
    ax.set_title(col[0].replace(" Train","") + ' Comparison')
plt.tight_layout()
plt.show()

# Cross-Validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
cv_results = {}
for model_name , model in tqdm(models.items()):
    X_cv , y_cv = get_fit_data(model_name)
    scores = cross_validate(model, X_cv, y_cv, cv=cv, scoring=['f1','roc_auc'], n_jobs=-1)
    cv_results[model_name] = {
        "CV F1 mean":scores['test_f1'].mean(),
        "CV F1 std":scores['test_f1'].std(),
        "CV AUC mean":scores['test_roc_auc'].mean(),
        "CV AUC std":scores['test_roc_auc'].std()
    }

In [ ]:
cv_df = pd.DataFrame(cv_results).T.sort_values(by='CV F1 mean',ascending=False)
cv_df

In [ ]:
cv_df[['CV F1 mean','CV AUC mean']].plot.bar(figsize=(14,6))
plt.show()

# GridSearchCV

In [ ]:
param_grids = {
    "Logistic Regression": {"C": [0.01, 0.1, 1, 10]},
    "KNN": {"n_neighbors": [3, 5, 11, 21], "weights": ["uniform", "distance"]},
    "Naive Bayes": {"var_smoothing": [1e-09, 1e-07, 1e-05]},
    "SVM": {"C": [0.5, 1, 5]},
    "Decision Tree": {"max_depth": [3, 5, 10, 15], "min_samples_leaf": [1, 5, 20]},
    "Random Forest": {"n_estimators": [100, 200], "max_depth": [10, 20], "min_samples_leaf": [1, 5]},
    "ExtraTrees": {"n_estimators": [100, 200], "max_depth": [10, None]},
    "Gradient Boosting": {"n_estimators": [100, 200], "learning_rate": [0.05, 0.1], "max_depth": [3, 5]},
    "AdaBoost": {"n_estimators": [50, 100, 200], "learning_rate": [0.1, 1]},
    "XGBoost": {"n_estimators": [100, 300], "learning_rate": [0.05, 0.1], "max_depth": [3, 6]},
    "LightGBM": {"n_estimators": [100, 300], "learning_rate": [0.05, 0.1], "num_leaves": [15, 31]}
}

In [ ]:
candidates = [name for name in cv_df.index if name in param_grids][:3]
candidates

In [ ]:
grids = {}
grid_rows = {}

for name in candidates:
    X_g , y_g = get_fit_data(name)
    grid = GridSearchCV(estimator=models[name], param_grid=param_grids[name], cv=cv, scoring='f1', n_jobs=-1)
    grid.fit(X_g, y_g)
    grids[name] = grid

    print(name)
    print("best_params_ :", grid.best_params_)
    print("best_score_  :", round(grid.best_score_,4))
    print()

    grid_rows[name] = {
        "CV F1 before":cv_df.loc[name,"CV F1 mean"],
        "CV F1 after (best_score_)":grid.best_score_,
        "best_params":grid.best_params_
    }

In [ ]:
grid_df = pd.DataFrame(grid_rows).T
grid_df["Improvement"] = grid_df["CV F1 after (best_score_)"] - grid_df["CV F1 before"]
grid_df

In [ ]:
best_name = grid_df["CV F1 after (best_score_)"].astype(float).idxmax()
best_params = grids[best_name].best_params_
best_tuned = grids[best_name].best_estimator_
best_default = models[best_name]
best_name , best_params

# Final Evaluation on Test Set

In [ ]:
rows = {}
for label , m in [("Before tuning",best_default),("After tuning",best_tuned)]:
    Acc, Pre, Rec, F1, AUC = classification_metrics(m, X_test, y_test)
    rows[label] = {"Accuracy":Acc,"Precision":Pre,"Recall":Rec,"F1":F1,"ROC AUC":AUC}

final_df = pd.DataFrame(rows).T
final_df

In [ ]:
y_pred = best_tuned.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm,annot=True,fmt="d",cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix - " + best_name)
plt.show()

In [ ]:
fig , ax = plt.subplots(1,2,figsize=(14,5))
RocCurveDisplay.from_estimator(best_default, X_test, y_test, name="Before tuning", ax=ax[0])
RocCurveDisplay.from_estimator(best_tuned, X_test, y_test, name="After tuning", ax=ax[0])
ax[0].plot([0,1],[0,1],"k--")
ax[0].set_title("ROC Curve")
PrecisionRecallDisplay.from_estimator(best_default, X_test, y_test, name="Before tuning", ax=ax[1])
PrecisionRecallDisplay.from_estimator(best_tuned, X_test, y_test, name="After tuning", ax=ax[1])
ax[1].set_title("Precision-Recall Curve")
plt.show()

In [ ]:
if hasattr(best_tuned, "predict_proba"):
    y_score = best_tuned.predict_proba(X_test)[:,1]
else:
    y_score = best_tuned.decision_function(X_test)

print("ROC AUC :", round(roc_auc_score(y_test, y_score),4))
print("Average Precision (AP) :", round(average_precision_score(y_test, y_score),4))
print("Baseline AP (share of positive class) :", round(y_test.mean(),4))

In [ ]:
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
print("Model:", best_name, best_params)
print("Actual positives:", tp+fn, "| detected:", tp, "| Recall:", round(tp/(tp+fn),3))
print("Predicted positives:", tp+fp, "| correct:", tp, "| Precision:", round(tp/(tp+fp),3) if (tp+fp) > 0 else 0)
print("False negatives:", fn, "| False positives:", fp)

# Feature Importance

In [ ]:
if hasattr(best_tuned, "feature_importances_"):
    imp = pd.Series(best_tuned.feature_importances_,index=X_train.columns)
elif hasattr(best_tuned, "coef_"):
    imp = pd.Series(np.abs(np.ravel(best_tuned.coef_)),index=X_train.columns)
else:
    imp = pd.Series(models["Random Forest"].feature_importances_,index=X_train.columns)
imp = imp.sort_values(ascending=False).head(15)
sns.barplot(x=imp.values,y=imp.index,orient="h")
plt.xlabel("Importance")
plt.title("Feature Importance - " + best_name)
plt.show()